In [ ]:
import pandas 
from pathlib import Path
import glob
import asyncio
import etl_books_async
import gitsource




# Question 1:

Get the lines of  'Think Python'. We build a ETL process to download and transform the pdfs into md. Results are stored int `books_text/`. 

#### Two-stage pipeline

- **Stage 1:** Concurrent PDF downloads *(I/O-bound)*
- **Stage 2:** Parallel PDF → Markdown conversion *(CPU-bound)*

---

#### Data source

- Reads `books.csv` from a remote URL
- Extracts PDF links from `pdf_url` *(fallback: `url`)*

---

#### Deterministic outputs

- PDFs saved in `pdfs/`
- Markdown files saved in `books_text/`
- Filenames derived from path in the folder

---

#### Concurrency model

- `asyncio` + `aiohttp` for high-throughput downloads
- `asyncio.Semaphore` to limit concurrent downloads
- `ProcessPoolExecutor` for safe parallel PDF conversion

---

#### Robustness

- Retry logic with exponential backoff for downloads
- Atomic file writes using temporary `.part` files
- Skips already-downloaded PDFs
- Skips already-converted Markdown files *(unless forced)*

---

#### Fault tolerance

- Uses `asyncio.gather(..., return_exceptions=True)`
- Continues processing even if some files fail
- Reports success/failure counts per stage

---

#### Performance-aware

- Separates I/O-bound and CPU-bound workloads
- Concurrency levels configurable:
  - `DOWNLOAD_CONCURRENCY`
  - `CONVERT_CONCURRENCY`
- Measures total execution time



In [ ]:
# Run the etl 

await etl_books_async.run_etl_async(force_md=False)


In [35]:
from pathlib import Path


md_dir = Path("books_text")

for md_file in md_dir.glob("thinkpython*.md"):
    print(f"The python book is: {md_file.name}")

    lines = md_file.read_text(encoding="utf-8").splitlines()
    content = "\n".join(lines)

    print(f"Content length (chars): {len(content)}")

    lines_cleaned = [line for line in lines if line.strip()]

    print(lines[:5])
    print(lines_cleaned[:5])

    print(".........")
    print(
        f"There are {len(lines)} lines. "
        f"If we remove empty lines, there are {len(lines_cleaned)} lines."
    )

The python book is: thinkpython2.md
Content length (chars): 480870
['## Think Python', '', '#### How to Think Like a Computer Scientist', '', '2nd Edition, Version 2.4.0']
['## Think Python', '#### How to Think Like a Computer Scientist', '2nd Edition, Version 2.4.0', '## Think Python', '#### How to Think Like a Computer Scientist']
.........
There are 16604 lines. If we remove empty lines, there are 9940 lines.


# Question 2. Chunking for RAG

In [88]:
books_path = Path('books_text')


def prepare_document(md_file: Path) -> dict:
    # 1) Read file
    text = md_file.read_text(encoding="utf-8")

    # 2) Split into lines
    lines = text.splitlines()

    # 3) Remove empty/whitespace-only lines
    lines_cleaned = [line.strip() for line in lines if line.strip()]

    # 4) Join lines into one large string separated by \n
    content = "\n".join(lines_cleaned)

    # 5) Build dict
    return {
        "source": md_file.name,
        "content": content,
    }


def create_docs(md_dir: Path) -> list[dict]:
    return [prepare_document(p) for p in md_dir.glob("*.md")]

docs = create_docs(books_path)

books = []
for doc in docs:
    print(doc.get("source"))
    books.append(doc.get("source"))
    

Think-C.md
PhysicalModelingInMatlab4.md
thinkos.md
thinkcomplexity2.md
thinkdsp.md
thinkjava2.md
thinkpython2.md


In [97]:
docs_by_source = {doc["source"]: doc for doc in docs}

content_python = docs_by_source["thinkpython2.md"]
content_python


{'source': 'thinkpython2.md',
 'content': '## Think Python\n#### How to Think Like a Computer Scientist\n2nd Edition, Version 2.4.0\n## Think Python\n#### How to Think Like a Computer Scientist\n2nd Edition, Version 2.4.0\n#### Allen Downey Green Tea Press\nNeedham, Massachusetts\nCopyright © 2015 Allen Downey.\nGreen Tea Press\n9 Washburn Ave\nNeedham MA 02492\nPermission is granted to copy, distribute, and/or modify this document under the terms of the\nCreative Commons Attribution-NonCommercial 3.0 Unported License, which is available at `[http:](http://creativecommons.org/licenses/by-nc/3.0/)`\n`[//creativecommons.org/licenses/by-nc/3.0/](http://creativecommons.org/licenses/by-nc/3.0/)` .\nThe original form of this book is L [A] TEX source code. Compiling this L [A] TEX source has the effect of generating a device-independent representation of a textbook, which can be converted to other formats\nand printed.\nThe L [A] TEX source for this book is available from `[http://www.thinkpy

In [111]:
from gitsource import chunk_documents

python_chunks = chunk_documents([content_python], size = 100, step = 50, content_field_name="content")

python_chunks[0:10]  # Show first two chunks

print(type(content_python['content']))
print(len(python_chunks[0]["content"]))

for c in python_chunks[:3]:
    print(c["start"], len(c["content"]), repr(c["content"][:40]))
    
print("Total chunks:", len(python_chunks))

<class 'str'>
100
0 100 '## Think Python\n#### How to Think Like a'
50 100 'Scientist\n2nd Edition, Version 2.4.0\n## '
100 100 'on\n#### How to Think Like a Computer Sci'
Total chunks: 9428
